<a href="https://colab.research.google.com/github/seop-byte/worldcup-2026-winner-prediction/blob/main/worldcup_2026_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 FIFA World Cup Winner Prediction

## Project Goal
This project aims to predict the winning probability of national teams in the 2026 FIFA World Cup using player market values, historical World Cup records, and tournament simulation.

## Problem Definition
Since the final result of the 2026 World Cup is unknown, this project treats the prediction task as a simulation-based machine learning problem. Team strength is estimated using player market values and historical performance records, and match results are simulated based on relative team strength.

## Main Steps
1. Load player and historical World Cup data
2. Preprocess market values and positions
3. Calculate national team strength
4. Build baseline and improved strength models
5. Simulate the 2026 tournament format
6. Estimate winning probability through repeated simulations
7. Compare results across different models

## Expected Output
- Team strength ranking
- Match win probability
- World Cup tournament simulation results
- Predicted winning probability for each country

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import Counter

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
!pip install kagglehub

In [8]:
# 1. 필요한 라이브러리
import pandas as pd
import numpy as np
import os

# 2. 파일 업로드
from google.colab import files
uploaded = files.upload()

# 3. CSV 파일 읽기
matches = pd.read_csv("all_matches.csv")
countries = pd.read_csv("countries_names.csv")
players = pd.read_csv("players.csv")
valuations = pd.read_csv("player_valuations.csv")

# 4. 데이터 크기 확인
print("matches:", matches.shape)
print("countries:", countries.shape)
print("players:", players.shape)
print("valuations:", valuations.shape)

# 5. 컬럼 확인
print("\n[matches columns]")
print(matches.columns.tolist())

print("\n[countries columns]")
print(countries.columns.tolist())

print("\n[players columns]")
print(players.columns.tolist())

print("\n[valuations columns]")
print(valuations.columns.tolist())

# 6. 앞부분 확인
display(matches.head())
display(countries.head())
display(players.head())
display(valuations.head())

Saving all_matches.csv to all_matches.csv
Saving countries_names.csv to countries_names.csv
Saving player_valuations.csv to player_valuations.csv
Saving players.csv to players.csv
matches: (51492, 8)
countries: (289, 3)
players: (47669, 26)
valuations: (640176, 6)

[matches columns]
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'country', 'neutral']

[countries columns]
['original_name', 'current_name', 'color_code']

[players columns]
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

[valuations columns]
['player_id', 'date', 'mark

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


,original_name,current_name,color_code
0,Afghanistan,Afghanistan,#AF2B34
1,Albania,Albania,#E41A13
2,Algeria,Algeria,#006233
3,Andorra,Andorra,#FFB800
4,Angola,Angola,#FF0000


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


In [9]:
# 날짜 형식 변환
matches["date"] = pd.to_datetime(matches["date"])
valuations["date"] = pd.to_datetime(valuations["date"])

# 국가명 통일용 dictionary 만들기
country_map = dict(zip(countries["original_name"], countries["current_name"]))

# all_matches의 국가명 통일
matches["home_team"] = matches["home_team"].replace(country_map)
matches["away_team"] = matches["away_team"].replace(country_map)
matches["country"] = matches["country"].replace(country_map)

# players의 국적도 국가명 통일
players["country_of_citizenship"] = players["country_of_citizenship"].replace(country_map)

print("Date converted and country names normalized.")
display(matches.head())
display(players[["player_id", "name", "country_of_citizenship", "position", "sub_position"]].head())

Date converted and country names normalized.


,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


,player_id,name,country_of_citizenship,position,sub_position
0,10,Miroslav Klose,Germany,Attack,Centre-Forward
1,26,Roman Weidenfeller,Germany,Goalkeeper,Goalkeeper
2,65,Dimitar Berbatov,Bulgaria,Attack,Centre-Forward
3,77,Lúcio,Brazil,Defender,Centre-Back
4,80,Tom Starke,Germany,Goalkeeper,Goalkeeper


In [10]:
# 최근 5개 월드컵 기준 직전 기간 설정
# 각 월드컵 시작 전까지의 A매치 성적을 feature로 사용

world_cup_periods = {
    2006: ("2002-07-01", "2006-06-08"),
    2010: ("2006-07-10", "2010-06-10"),
    2014: ("2010-07-12", "2014-06-11"),
    2018: ("2014-07-14", "2018-06-13"),
    2022: ("2018-07-16", "2022-11-19")
}

def make_team_stats(matches_df, start_date, end_date, wc_year):
    period_matches = matches_df[
        (matches_df["date"] >= start_date) &
        (matches_df["date"] <= end_date)
    ].copy()

    # 홈팀 기준
    home = period_matches[[
        "date", "home_team", "away_team", "home_score", "away_score", "tournament"
    ]].copy()

    home = home.rename(columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_for",
        "away_score": "goals_against"
    })

    home["result"] = np.where(
        home["goals_for"] > home["goals_against"], "W",
        np.where(home["goals_for"] == home["goals_against"], "D", "L")
    )

    # 원정팀 기준
    away = period_matches[[
        "date", "home_team", "away_team", "home_score", "away_score", "tournament"
    ]].copy()

    away = away.rename(columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_for",
        "home_score": "goals_against"
    })

    away["result"] = np.where(
        away["goals_for"] > away["goals_against"], "W",
        np.where(away["goals_for"] == away["goals_against"], "D", "L")
    )

    # 홈/원정 합치기
    team_records = pd.concat([home, away], ignore_index=True)

    # 국가별 통계 계산
    team_stats = team_records.groupby("team").agg(
        matches_played=("result", "count"),
        wins=("result", lambda x: (x == "W").sum()),
        draws=("result", lambda x: (x == "D").sum()),
        losses=("result", lambda x: (x == "L").sum()),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum")
    ).reset_index()

    team_stats["goal_difference"] = team_stats["goals_for"] - team_stats["goals_against"]
    team_stats["win_rate"] = team_stats["wins"] / team_stats["matches_played"]
    team_stats["points"] = team_stats["wins"] * 3 + team_stats["draws"]
    team_stats["points_per_match"] = team_stats["points"] / team_stats["matches_played"]
    team_stats["goals_for_per_match"] = team_stats["goals_for"] / team_stats["matches_played"]
    team_stats["goals_against_per_match"] = team_stats["goals_against"] / team_stats["matches_played"]
    team_stats["world_cup_year"] = wc_year

    return team_stats

In [11]:
# 5개 월드컵에 대해 국가별 직전 4년 성적 계산

all_team_stats = []

for wc_year, (start_date, end_date) in world_cup_periods.items():
    temp = make_team_stats(matches, start_date, end_date, wc_year)
    all_team_stats.append(temp)

all_team_stats = pd.concat(all_team_stats, ignore_index=True)

print("전체 국가-월드컵 성적 데이터 크기:", all_team_stats.shape)
display(all_team_stats.head(20))

전체 국가-월드컵 성적 데이터 크기: (1121, 14)


,team,matches_played,wins,draws,losses,goals_for,goals_against,goal_difference,win_rate,points,points_per_match,goals_for_per_match,goals_against_per_match,world_cup_year
0,Afghanistan,14,2,2,10,8,44,-36,0.142857,8,0.571429,0.571429,3.142857,2006
1,Albania,34,12,5,17,43,54,-11,0.352941,41,1.205882,1.264706,1.588235,2006
2,Algeria,43,14,15,14,49,51,-2,0.325581,57,1.325581,1.139535,1.186047,2006
3,American Samoa,4,0,0,4,1,34,-33,0.000000,0,0.000000,0.250000,8.500000,2006
4,Andorra,25,1,3,21,5,65,-60,0.040000,6,0.240000,0.200000,2.600000,2006
5,Angola,47,19,14,14,61,50,11,0.404255,71,1.510638,1.297872,1.063830,2006
6,Anguilla,4,0,1,3,1,10,-9,0.000000,1,0.250000,0.250000,2.500000,2006
7,Antigua and Barbuda,19,5,2,12,21,37,-16,0.263158,17,0.894737,1.105263,1.947368,2006
8,Argentina,50,30,10,10,94,55,39,0.600000,100,2.000000,1.880000,1.100000,2006
9,Armenia,29,6,4,19,23,55,-32,0.206897,22,0.758621,0.793103,1.896552,2006


In [12]:
# 월드컵 연도별 국가 수 확인

display(
    all_team_stats.groupby("world_cup_year")["team"]
    .nunique()
    .reset_index(name="num_teams")
)

,world_cup_year,num_teams
0,2006,225
1,2010,221
2,2014,228
3,2018,224
4,2022,223


In [13]:
# 월드컵별 시장가치 기준일
# 각 월드컵 시작 전날 또는 시작 직전 날짜를 기준으로 잡음

wc_cutoff_dates = {
    2006: "2006-06-08",
    2010: "2010-06-10",
    2014: "2014-06-11",
    2018: "2018-06-13",
    2022: "2022-11-19"
}

# 선수 데이터와 시장가치 데이터 합치기
player_value_data = valuations.merge(
    players[["player_id", "name", "country_of_citizenship", "position", "sub_position"]],
    on="player_id",
    how="left"
)

# 확인
print(player_value_data.shape)
display(player_value_data.head())

(640176, 10)


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,name,country_of_citizenship,position,sub_position
0,405973,2000-01-20,150000,Unknown,3057,BE1,Fadel Gobitaka,Togo,Attack,Left Winger
1,342216,2001-07-20,100000,Unknown,1241,SC1,Julien Serrano,France,Defender,Left-Back
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1,Florin Cernat,Romania,Midfield,Attacking Midfield
3,6893,2003-12-15,900000,Galatasaray,984,GB1,Gabriel Tamas,Romania,Defender,Centre-Back
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1,Miroslav Klose,Germany,Attack,Centre-Forward


In [14]:
# 포지션을 4-3-3 기준 그룹으로 단순화하는 함수

def position_group(position):
    if position == "Goalkeeper":
        return "GK"
    elif position == "Defender":
        return "DF"
    elif position == "Midfield":
        return "MF"
    elif position == "Attack":
        return "FW"
    else:
        return "OTHER"

player_value_data["position_group"] = player_value_data["position"].apply(position_group)

display(player_value_data[["name", "country_of_citizenship", "position", "sub_position", "position_group", "market_value_in_eur", "date"]].head(20))

,name,country_of_citizenship,position,sub_position,position_group,market_value_in_eur,date
0,Fadel Gobitaka,Togo,Attack,Left Winger,FW,150000,2000-01-20
1,Julien Serrano,France,Defender,Left-Back,DF,100000,2001-07-20
2,Florin Cernat,Romania,Midfield,Attacking Midfield,MF,400000,2003-12-09
3,Gabriel Tamas,Romania,Defender,Centre-Back,DF,900000,2003-12-15
4,Miroslav Klose,Germany,Attack,Centre-Forward,FW,7000000,2004-10-04
5,Roman Weidenfeller,Germany,Goalkeeper,Goalkeeper,GK,1500000,2004-10-04
6,Dimitar Berbatov,Bulgaria,Attack,Centre-Forward,FW,8000000,2004-10-04
7,Lúcio,Brazil,Defender,Centre-Back,DF,13000000,2004-10-04
8,Tom Starke,Germany,Goalkeeper,Goalkeeper,GK,400000,2004-10-04
9,Dedê,Brazil,Defender,Left-Back,DF,9500000,2004-10-04


In [15]:
# 특정 월드컵 기준일 이전의 선수별 최신 시장가치 가져오기

def get_latest_values_before_cutoff(df, cutoff_date, wc_year):
    cutoff_date = pd.to_datetime(cutoff_date)

    temp = df[df["date"] <= cutoff_date].copy()

    # 선수별로 기준일 이전 가장 최근 시장가치만 남김
    temp = temp.sort_values(["player_id", "date"])
    latest = temp.groupby("player_id").tail(1).copy()

    latest["world_cup_year"] = wc_year

    return latest

In [16]:
# 월드컵별 국가별 4-3-3 베스트11 시장가치 합 계산

def calculate_best11_market_value(latest_values, wc_year):
    result_rows = []

    for country, group in latest_values.groupby("country_of_citizenship"):
        gk = group[group["position_group"] == "GK"].nlargest(1, "market_value_in_eur")
        df = group[group["position_group"] == "DF"].nlargest(4, "market_value_in_eur")
        mf = group[group["position_group"] == "MF"].nlargest(3, "market_value_in_eur")
        fw = group[group["position_group"] == "FW"].nlargest(3, "market_value_in_eur")

        best11 = pd.concat([gk, df, mf, fw])

        # 4-3-3을 완성할 수 있는 국가만 사용
        if len(gk) == 1 and len(df) == 4 and len(mf) == 3 and len(fw) == 3:
            result_rows.append({
                "team": country,
                "world_cup_year": wc_year,
                "best11_market_value": best11["market_value_in_eur"].sum(),
                "avg_best11_market_value": best11["market_value_in_eur"].mean(),
                "num_best11_players": len(best11),
                "gk_value": gk["market_value_in_eur"].sum(),
                "df_value": df["market_value_in_eur"].sum(),
                "mf_value": mf["market_value_in_eur"].sum(),
                "fw_value": fw["market_value_in_eur"].sum()
            })

    return pd.DataFrame(result_rows)


all_market_values = []

for wc_year, cutoff_date in wc_cutoff_dates.items():
    latest_values = get_latest_values_before_cutoff(player_value_data, cutoff_date, wc_year)
    market_df = calculate_best11_market_value(latest_values, wc_year)
    all_market_values.append(market_df)

all_market_values = pd.concat(all_market_values, ignore_index=True)

print("시장가치 데이터 크기:", all_market_values.shape)
display(all_market_values.head(20))

# 연도별 시장가치 계산 가능한 국가 수
display(
    all_market_values.groupby("world_cup_year")["team"]
    .nunique()
    .reset_index(name="num_teams_with_market_value")
)

시장가치 데이터 크기: (416, 9)


,team,world_cup_year,best11_market_value,avg_best11_market_value,num_best11_players,gk_value,df_value,mf_value,fw_value
0,Argentina,2006,145000000,1.318182e+07,11,6500000,46000000,51000000,41500000
1,Austria,2006,23650000,2.150000e+06,11,1400000,10000000,8200000,4050000
2,Belgium,2006,63000000,5.727273e+06,11,3500000,36500000,11000000,12000000
3,Brazil,2006,171500000,1.559091e+07,11,6000000,59000000,69000000,37500000
4,Cameroon,2006,66900000,6.081818e+06,11,4000000,7600000,13500000,41800000
5,Cote d'Ivoire,2006,78250000,7.113636e+06,11,900000,18850000,18000000,40500000
6,Croatia,2006,45200000,4.109091e+06,11,2500000,10750000,10700000,21250000
7,Czech Republic,2006,72950000,6.631818e+06,11,22000000,18200000,22000000,10750000
8,Denmark,2006,57200000,5.200000e+06,11,6200000,20500000,20000000,10500000
9,England,2006,261100000,2.373636e+07,11,7850000,85250000,87000000,81000000


,world_cup_year,num_teams_with_market_value
0,2006,27
1,2010,61
2,2014,89
3,2018,109
4,2022,130


In [17]:
# 월드컵 본선 참가국 리스트
# 일단 시장가치 데이터가 상대적으로 충분한 2010, 2014, 2018, 2022 사용

world_cup_teams = {
    2010: [
        "South Africa", "Mexico", "Uruguay", "France",
        "Argentina", "Nigeria", "South Korea", "Greece",
        "England", "United States", "Algeria", "Slovenia",
        "Germany", "Australia", "Serbia", "Ghana",
        "Netherlands", "Denmark", "Japan", "Cameroon",
        "Italy", "Paraguay", "New Zealand", "Slovakia",
        "Brazil", "North Korea", "Ivory Coast", "Portugal",
        "Spain", "Switzerland", "Honduras", "Chile"
    ],
    2014: [
        "Brazil", "Croatia", "Mexico", "Cameroon",
        "Spain", "Netherlands", "Chile", "Australia",
        "Colombia", "Greece", "Ivory Coast", "Japan",
        "Uruguay", "Costa Rica", "England", "Italy",
        "Switzerland", "Ecuador", "France", "Honduras",
        "Argentina", "Bosnia-Herzegovina", "Iran", "Nigeria",
        "Germany", "Portugal", "Ghana", "United States",
        "Belgium", "Algeria", "Russia", "South Korea"
    ],
    2018: [
        "Russia", "Saudi Arabia", "Egypt", "Uruguay",
        "Portugal", "Spain", "Morocco", "Iran",
        "France", "Australia", "Peru", "Denmark",
        "Argentina", "Iceland", "Croatia", "Nigeria",
        "Brazil", "Switzerland", "Costa Rica", "Serbia",
        "Germany", "Mexico", "Sweden", "South Korea",
        "Belgium", "Panama", "Tunisia", "England",
        "Poland", "Senegal", "Colombia", "Japan"
    ],
    2022: [
        "Qatar", "Ecuador", "Senegal", "Netherlands",
        "England", "Iran", "United States", "Wales",
        "Argentina", "Saudi Arabia", "Mexico", "Poland",
        "France", "Australia", "Denmark", "Tunisia",
        "Spain", "Costa Rica", "Germany", "Japan",
        "Belgium", "Canada", "Morocco", "Croatia",
        "Brazil", "Serbia", "Switzerland", "Cameroon",
        "Portugal", "Ghana", "Uruguay", "South Korea"
    ]
}

# 참가국 리스트를 DataFrame으로 변환
wc_team_rows = []

for year, teams in world_cup_teams.items():
    for team in teams:
        wc_team_rows.append({
            "world_cup_year": year,
            "team": team
        })

wc_teams_df = pd.DataFrame(wc_team_rows)

print(wc_teams_df.shape)
display(wc_teams_df.head())

(128, 2)


,world_cup_year,team
0,2010,South Africa
1,2010,Mexico
2,2010,Uruguay
3,2010,France
4,2010,Argentina


In [18]:
# 월드컵 참가국 기준으로 경기 성적 데이터 필터링

team_stats_wc = wc_teams_df.merge(
    all_team_stats,
    on=["world_cup_year", "team"],
    how="left"
)

print("월드컵 참가국 경기 성적 데이터:", team_stats_wc.shape)
display(team_stats_wc.head(20))

# 결측치 확인
display(team_stats_wc.isnull().sum())

월드컵 참가국 경기 성적 데이터: (128, 14)


,world_cup_year,team,matches_played,wins,draws,losses,goals_for,goals_against,goal_difference,win_rate,points,points_per_match,goals_for_per_match,goals_against_per_match
0,2010,South Africa,76.0,35.0,18.0,23.0,96.0,59.0,37.0,0.460526,123.0,1.618421,1.263158,0.776316
1,2010,Mexico,71.0,42.0,10.0,19.0,133.0,67.0,66.0,0.591549,136.0,1.915493,1.873239,0.943662
2,2010,Uruguay,43.0,19.0,13.0,11.0,71.0,47.0,24.0,0.441860,70.0,1.627907,1.651163,1.093023
3,2010,France,48.0,26.0,11.0,11.0,67.0,37.0,30.0,0.541667,89.0,1.854167,1.395833,0.770833
4,2010,Argentina,47.0,28.0,8.0,11.0,81.0,46.0,35.0,0.595745,92.0,1.957447,1.723404,0.978723
5,2010,Nigeria,45.0,24.0,13.0,8.0,58.0,29.0,29.0,0.533333,85.0,1.888889,1.288889,0.644444
6,2010,South Korea,58.0,27.0,20.0,11.0,85.0,44.0,41.0,0.465517,101.0,1.741379,1.465517,0.758621
7,2010,Greece,43.0,21.0,8.0,14.0,63.0,49.0,14.0,0.488372,71.0,1.651163,1.465116,1.139535
8,2010,England,42.0,27.0,6.0,9.0,93.0,32.0,61.0,0.642857,87.0,2.071429,2.214286,0.761905
9,2010,United States,61.0,36.0,6.0,19.0,112.0,74.0,38.0,0.590164,114.0,1.868852,1.836066,1.213115


,0
world_cup_year,0
team,0
matches_played,1
wins,1
draws,1
losses,1
goals_for,1
goals_against,1
goal_difference,1
win_rate,1


In [19]:
# 상대 수준을 반영한 국가별 성적 계산 함수로 다시 정의

def make_team_stats(matches_df, start_date, end_date, wc_year):
    period_matches = matches_df[
        (matches_df["date"] >= start_date) &
        (matches_df["date"] <= end_date)
    ].copy()

    # 홈팀 기준
    home = period_matches[[
        "date", "home_team", "away_team", "home_score", "away_score", "tournament"
    ]].copy()

    home = home.rename(columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_for",
        "away_score": "goals_against"
    })

    home["result"] = np.where(
        home["goals_for"] > home["goals_against"], "W",
        np.where(home["goals_for"] == home["goals_against"], "D", "L")
    )

    # 원정팀 기준
    away = period_matches[[
        "date", "home_team", "away_team", "home_score", "away_score", "tournament"
    ]].copy()

    away = away.rename(columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_for",
        "home_score": "goals_against"
    })

    away["result"] = np.where(
        away["goals_for"] > away["goals_against"], "W",
        np.where(away["goals_for"] == away["goals_against"], "D", "L")
    )

    # 홈/원정 합치기
    team_records = pd.concat([home, away], ignore_index=True)

    # 기본 승점
    team_records["result_points"] = np.where(
        team_records["result"] == "W", 3,
        np.where(team_records["result"] == "D", 1, 0)
    )

    # 먼저 각 팀의 기본 성적 계산
    basic_stats = team_records.groupby("team").agg(
        matches_played=("result", "count"),
        wins=("result", lambda x: (x == "W").sum()),
        draws=("result", lambda x: (x == "D").sum()),
        losses=("result", lambda x: (x == "L").sum()),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        points=("result_points", "sum")
    ).reset_index()

    basic_stats["goal_difference"] = basic_stats["goals_for"] - basic_stats["goals_against"]
    basic_stats["win_rate"] = basic_stats["wins"] / basic_stats["matches_played"]
    basic_stats["points_per_match"] = basic_stats["points"] / basic_stats["matches_played"]

    # 상대 강도 계산: 해당 기간의 경기당 승점을 0~1 사이로 정규화
    min_ppm = basic_stats["points_per_match"].min()
    max_ppm = basic_stats["points_per_match"].max()

    basic_stats["team_strength"] = (
        (basic_stats["points_per_match"] - min_ppm) /
        (max_ppm - min_ppm)
    )

    # 각 경기별 상대 팀 강도 붙이기
    strength_map = basic_stats.set_index("team")["team_strength"].to_dict()
    team_records["opponent_strength"] = team_records["opponent"].map(strength_map)

    # 혹시 매칭 안 되는 상대는 평균값으로 대체
    team_records["opponent_strength"] = team_records["opponent_strength"].fillna(
        team_records["opponent_strength"].mean()
    )

    # 상대 강도 보정 계수
    # 약팀 상대: 0.5배 근처
    # 중간 팀 상대: 1.25배 근처
    # 강팀 상대: 최대 2.0배
    team_records["strength_multiplier"] = 0.5 + 1.5 * team_records["opponent_strength"]

    # 보정 승점
    team_records["weighted_points"] = (
        team_records["result_points"] * team_records["strength_multiplier"]
    )

    # 최종 팀 통계
    team_stats = team_records.groupby("team").agg(
        matches_played=("result", "count"),
        wins=("result", lambda x: (x == "W").sum()),
        draws=("result", lambda x: (x == "D").sum()),
        losses=("result", lambda x: (x == "L").sum()),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        points=("result_points", "sum"),
        avg_opponent_strength=("opponent_strength", "mean"),
        weighted_points=("weighted_points", "sum")
    ).reset_index()

    team_stats["goal_difference"] = team_stats["goals_for"] - team_stats["goals_against"]
    team_stats["win_rate"] = team_stats["wins"] / team_stats["matches_played"]
    team_stats["points_per_match"] = team_stats["points"] / team_stats["matches_played"]
    team_stats["weighted_points_per_match"] = team_stats["weighted_points"] / team_stats["matches_played"]
    team_stats["goals_for_per_match"] = team_stats["goals_for"] / team_stats["matches_played"]
    team_stats["goals_against_per_match"] = team_stats["goals_against"] / team_stats["matches_played"]
    team_stats["world_cup_year"] = wc_year

    return team_stats

print("보정승점 포함 make_team_stats 함수 정의 완료")

보정승점 포함 make_team_stats 함수 정의 완료


In [20]:
# 보정승점이 들어간 all_team_stats 다시 생성

all_team_stats = []

for wc_year, (start_date, end_date) in world_cup_periods.items():
    temp = make_team_stats(matches, start_date, end_date, wc_year)
    all_team_stats.append(temp)

all_team_stats = pd.concat(all_team_stats, ignore_index=True)

print("전체 국가-월드컵 성적 데이터 크기:", all_team_stats.shape)
display(all_team_stats.head(20))

display(
    all_team_stats.groupby("world_cup_year")["team"]
    .nunique()
    .reset_index(name="num_teams")
)

전체 국가-월드컵 성적 데이터 크기: (1121, 17)


,team,matches_played,wins,draws,losses,goals_for,goals_against,points,avg_opponent_strength,weighted_points,goal_difference,win_rate,points_per_match,weighted_points_per_match,goals_for_per_match,goals_against_per_match,world_cup_year
0,Afghanistan,14,2,2,10,8,44,8,0.423855,8.853936,-36,0.142857,0.571429,0.632424,0.571429,3.142857,2006
1,Albania,34,12,5,17,43,54,41,0.510788,45.524517,-11,0.352941,1.205882,1.338956,1.264706,1.588235,2006
2,Algeria,43,14,15,14,49,51,57,0.528632,66.788481,-2,0.325581,1.325581,1.553220,1.139535,1.186047,2006
3,American Samoa,4,0,0,4,1,34,0,0.496295,0.000000,-33,0.000000,0.000000,0.000000,0.250000,8.500000,2006
4,Andorra,25,1,3,21,5,65,6,0.603247,7.313241,-60,0.040000,0.240000,0.292530,0.200000,2.600000,2006
5,Angola,47,19,14,14,61,50,71,0.536111,86.357449,11,0.404255,1.510638,1.837393,1.297872,1.063830,2006
6,Anguilla,4,0,1,3,1,10,1,0.435897,1.076923,-9,0.000000,0.250000,0.269231,0.250000,2.500000,2006
7,Antigua and Barbuda,19,5,2,12,21,37,17,0.539411,20.197368,-16,0.263158,0.894737,1.063019,1.105263,1.947368,2006
8,Argentina,50,30,10,10,94,55,100,0.600089,136.290147,39,0.600000,2.000000,2.725803,1.880000,1.100000,2006
9,Armenia,29,6,4,19,23,55,22,0.559688,21.579200,-32,0.206897,0.758621,0.744110,0.793103,1.896552,2006


,world_cup_year,num_teams
0,2006,225
1,2010,221
2,2014,228
3,2018,224
4,2022,223


In [21]:
# 월드컵 참가국 기준으로 경기 성적 데이터 다시 필터링

team_stats_wc = wc_teams_df.merge(
    all_team_stats,
    on=["world_cup_year", "team"],
    how="left"
)

print("월드컵 참가국 경기 성적 데이터:", team_stats_wc.shape)
display(team_stats_wc.head(20))

# 결측치 확인
display(team_stats_wc.isnull().sum())

월드컵 참가국 경기 성적 데이터: (128, 17)


,world_cup_year,team,matches_played,wins,draws,losses,goals_for,goals_against,points,avg_opponent_strength,weighted_points,goal_difference,win_rate,points_per_match,weighted_points_per_match,goals_for_per_match,goals_against_per_match
0,2010,South Africa,76.0,35.0,18.0,23.0,96.0,59.0,123.0,0.468901,138.235364,37.0,0.460526,1.618421,1.818886,1.263158,0.776316
1,2010,Mexico,71.0,42.0,10.0,19.0,133.0,67.0,136.0,0.502713,164.223304,66.0,0.591549,1.915493,2.313004,1.873239,0.943662
2,2010,Uruguay,43.0,19.0,13.0,11.0,71.0,47.0,70.0,0.516465,87.992515,24.0,0.441860,1.627907,2.046338,1.651163,1.093023
3,2010,France,48.0,26.0,11.0,11.0,67.0,37.0,89.0,0.503080,102.995548,30.0,0.541667,1.854167,2.145741,1.395833,0.770833
4,2010,Argentina,47.0,28.0,8.0,11.0,81.0,46.0,92.0,0.546102,118.688944,35.0,0.595745,1.957447,2.525297,1.723404,0.978723
5,2010,Nigeria,45.0,24.0,13.0,8.0,58.0,29.0,85.0,0.465793,95.114556,29.0,0.533333,1.888889,2.113657,1.288889,0.644444
6,2010,South Korea,58.0,27.0,20.0,11.0,85.0,44.0,101.0,0.527032,124.634078,41.0,0.465517,1.741379,2.148863,1.465517,0.758621
7,2010,Greece,43.0,21.0,8.0,14.0,63.0,49.0,71.0,0.476848,78.466285,14.0,0.488372,1.651163,1.824797,1.465116,1.139535
8,2010,England,42.0,27.0,6.0,9.0,93.0,32.0,87.0,0.536756,103.184062,61.0,0.642857,2.071429,2.456763,2.214286,0.761905
9,2010,United States,61.0,36.0,6.0,19.0,112.0,74.0,114.0,0.545739,144.286373,38.0,0.590164,1.868852,2.365350,1.836066,1.213115


,0
world_cup_year,0
team,0
matches_played,1
wins,1
draws,1
losses,1
goals_for,1
goals_against,1
points,1
avg_opponent_strength,1


In [22]:
# 월드컵 참가국 기준으로 시장가치 데이터 다시 붙이기

final_base = team_stats_wc.merge(
    all_market_values,
    on=["world_cup_year", "team"],
    how="left"
)

print("성적 + 시장가치 데이터:", final_base.shape)
display(final_base.head(20))

# 결측치 확인
display(final_base.isnull().sum())

성적 + 시장가치 데이터: (128, 24)


,world_cup_year,team,matches_played,wins,draws,losses,goals_for,goals_against,points,avg_opponent_strength,...,weighted_points_per_match,goals_for_per_match,goals_against_per_match,best11_market_value,avg_best11_market_value,num_best11_players,gk_value,df_value,mf_value,fw_value
0,2010,South Africa,76.0,35.0,18.0,23.0,96.0,59.0,123.0,0.468901,...,1.818886,1.263158,0.776316,13050000.0,1.186364e+06,11.0,150000.0,2200000.0,9450000.0,1250000.0
1,2010,Mexico,71.0,42.0,10.0,19.0,133.0,67.0,136.0,0.502713,...,2.313004,1.873239,0.943662,55500000.0,5.045455e+06,11.0,8000000.0,18000000.0,14500000.0,15000000.0
2,2010,Uruguay,43.0,19.0,13.0,11.0,71.0,47.0,70.0,0.516465,...,2.046338,1.651163,1.093023,100800000.0,9.163636e+06,11.0,7500000.0,35200000.0,21600000.0,36500000.0
3,2010,France,48.0,26.0,11.0,11.0,67.0,37.0,89.0,0.503080,...,2.145741,1.395833,0.770833,275000000.0,2.500000e+07,11.0,16000000.0,83000000.0,68000000.0,108000000.0
4,2010,Argentina,47.0,28.0,8.0,11.0,81.0,46.0,92.0,0.546102,...,2.525297,1.723404,0.978723,273500000.0,2.486364e+07,11.0,6000000.0,44500000.0,75000000.0,148000000.0
5,2010,Nigeria,45.0,24.0,13.0,8.0,58.0,29.0,85.0,0.465793,...,2.113657,1.288889,0.644444,86500000.0,7.863636e+06,11.0,2000000.0,28500000.0,21000000.0,35000000.0
6,2010,South Korea,58.0,27.0,20.0,11.0,85.0,44.0,101.0,0.527032,...,2.148863,1.465517,0.758621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2010,Greece,43.0,21.0,8.0,14.0,63.0,49.0,71.0,0.476848,...,1.824797,1.465116,1.139535,60750000.0,5.522727e+06,11.0,2500000.0,23000000.0,21500000.0,13750000.0
8,2010,England,42.0,27.0,6.0,9.0,93.0,32.0,87.0,0.536756,...,2.456763,2.214286,0.761905,332000000.0,3.018182e+07,11.0,8000000.0,112000000.0,121000000.0,91000000.0
9,2010,United States,61.0,36.0,6.0,19.0,112.0,74.0,114.0,0.545739,...,2.365350,1.836066,1.213115,46200000.0,4.200000e+06,11.0,9500000.0,10500000.0,14000000.0,12200000.0


,0
world_cup_year,0
team,0
matches_played,1
wins,1
draws,1
losses,1
goals_for,1
goals_against,1
points,1
avg_opponent_strength,1


In [23]:
# 최종 feature 후보 컬럼 확인

print(final_base.columns.tolist())

display(final_base[[
    "world_cup_year",
    "team",
    "matches_played",
    "wins",
    "draws",
    "losses",
    "points_per_match",
    "avg_opponent_strength",
    "weighted_points_per_match",
    "best11_market_value"
]].head(20))

['world_cup_year', 'team', 'matches_played', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'points', 'avg_opponent_strength', 'weighted_points', 'goal_difference', 'win_rate', 'points_per_match', 'weighted_points_per_match', 'goals_for_per_match', 'goals_against_per_match', 'best11_market_value', 'avg_best11_market_value', 'num_best11_players', 'gk_value', 'df_value', 'mf_value', 'fw_value']


,world_cup_year,team,matches_played,wins,draws,losses,points_per_match,avg_opponent_strength,weighted_points_per_match,best11_market_value
0,2010,South Africa,76.0,35.0,18.0,23.0,1.618421,0.468901,1.818886,13050000.0
1,2010,Mexico,71.0,42.0,10.0,19.0,1.915493,0.502713,2.313004,55500000.0
2,2010,Uruguay,43.0,19.0,13.0,11.0,1.627907,0.516465,2.046338,100800000.0
3,2010,France,48.0,26.0,11.0,11.0,1.854167,0.503080,2.145741,275000000.0
4,2010,Argentina,47.0,28.0,8.0,11.0,1.957447,0.546102,2.525297,273500000.0
5,2010,Nigeria,45.0,24.0,13.0,8.0,1.888889,0.465793,2.113657,86500000.0
6,2010,South Korea,58.0,27.0,20.0,11.0,1.741379,0.527032,2.148863,NaN
7,2010,Greece,43.0,21.0,8.0,14.0,1.651163,0.476848,1.824797,60750000.0
8,2010,England,42.0,27.0,6.0,9.0,2.071429,0.536756,2.456763,332000000.0
9,2010,United States,61.0,36.0,6.0,19.0,1.868852,0.545739,2.365350,46200000.0


In [24]:
# 월드컵 실제 성적 label 만들기
# stage_score는 숫자가 클수록 좋은 성적
# 0: 조별리그 탈락
# 1: 16강
# 2: 8강
# 3: 4강
# 4: 준우승
# 5: 우승

world_cup_results = {
    2010: {
        "champion": ["Spain"],
        "runner_up": ["Netherlands"],
        "semi_final": ["Germany", "Uruguay"],
        "quarter_final": ["Argentina", "Brazil", "Ghana", "Paraguay"],
        "round_of_16": ["South Korea", "United States", "England", "Mexico",
                        "Slovakia", "Chile", "Portugal", "Japan"]
    },
    2014: {
        "champion": ["Germany"],
        "runner_up": ["Argentina"],
        "semi_final": ["Brazil", "Netherlands"],
        "quarter_final": ["Belgium", "Colombia", "Costa Rica", "France"],
        "round_of_16": ["Chile", "Uruguay", "Nigeria", "Algeria",
                        "Switzerland", "United States", "Mexico", "Greece"]
    },
    2018: {
        "champion": ["France"],
        "runner_up": ["Croatia"],
        "semi_final": ["Belgium", "England"],
        "quarter_final": ["Uruguay", "Brazil", "Sweden", "Russia"],
        "round_of_16": ["Argentina", "Portugal", "Spain", "Denmark",
                        "Mexico", "Japan", "Switzerland", "Colombia"]
    },
    2022: {
        "champion": ["Argentina"],
        "runner_up": ["France"],
        "semi_final": ["Croatia", "Morocco"],
        "quarter_final": ["Netherlands", "Brazil", "England", "Portugal"],
        "round_of_16": ["United States", "Australia", "Poland", "Senegal",
                        "Japan", "South Korea", "Spain", "Switzerland"]
    }
}

def get_stage_score(year, team):
    result = world_cup_results.get(year, {})

    if team in result.get("champion", []):
        return 5
    elif team in result.get("runner_up", []):
        return 4
    elif team in result.get("semi_final", []):
        return 3
    elif team in result.get("quarter_final", []):
        return 2
    elif team in result.get("round_of_16", []):
        return 1
    else:
        return 0

def get_stage_name(score):
    if score == 5:
        return "Champion"
    elif score == 4:
        return "Runner-up"
    elif score == 3:
        return "Semi-final"
    elif score == 2:
        return "Quarter-final"
    elif score == 1:
        return "Round of 16"
    else:
        return "Group stage"

final_base["stage_score"] = final_base.apply(
    lambda row: get_stage_score(row["world_cup_year"], row["team"]),
    axis=1
)

final_base["stage_name"] = final_base["stage_score"].apply(get_stage_name)

display(final_base[[
    "world_cup_year", "team", "stage_name", "stage_score",
    "points_per_match", "weighted_points_per_match",
    "best11_market_value"
]].head(40))

display(final_base["stage_name"].value_counts())

,world_cup_year,team,stage_name,stage_score,points_per_match,weighted_points_per_match,best11_market_value
0,2010,South Africa,Group stage,0,1.618421,1.818886,13050000.0
1,2010,Mexico,Round of 16,1,1.915493,2.313004,55500000.0
2,2010,Uruguay,Semi-final,3,1.627907,2.046338,100800000.0
3,2010,France,Group stage,0,1.854167,2.145741,275000000.0
4,2010,Argentina,Quarter-final,2,1.957447,2.525297,273500000.0
5,2010,Nigeria,Group stage,0,1.888889,2.113657,86500000.0
6,2010,South Korea,Round of 16,1,1.741379,2.148863,NaN
7,2010,Greece,Group stage,0,1.651163,1.824797,60750000.0
8,2010,England,Round of 16,1,2.071429,2.456763,332000000.0
9,2010,United States,Round of 16,1,1.868852,2.365350,46200000.0


,count
stage_name,
Group stage,64
Round of 16,32
Quarter-final,16
Semi-final,8
Runner-up,4
Champion,4


In [25]:
# 결측치 제거해서 최종 ML 데이터셋 만들기

ml_data = final_base.dropna().copy()

print("결측치 제거 전:", final_base.shape)
print("결측치 제거 후:", ml_data.shape)

display(ml_data.head())

# 최종 데이터 저장
ml_data.to_csv("worldcup_ml_dataset.csv", index=False)

print("worldcup_ml_dataset.csv 저장 완료")

결측치 제거 전: (128, 26)
결측치 제거 후: (115, 26)


,world_cup_year,team,matches_played,wins,draws,losses,goals_for,goals_against,points,avg_opponent_strength,...,goals_against_per_match,best11_market_value,avg_best11_market_value,num_best11_players,gk_value,df_value,mf_value,fw_value,stage_score,stage_name
0,2010,South Africa,76.0,35.0,18.0,23.0,96.0,59.0,123.0,0.468901,...,0.776316,13050000.0,1.186364e+06,11.0,150000.0,2200000.0,9450000.0,1250000.0,0,Group stage
1,2010,Mexico,71.0,42.0,10.0,19.0,133.0,67.0,136.0,0.502713,...,0.943662,55500000.0,5.045455e+06,11.0,8000000.0,18000000.0,14500000.0,15000000.0,1,Round of 16
2,2010,Uruguay,43.0,19.0,13.0,11.0,71.0,47.0,70.0,0.516465,...,1.093023,100800000.0,9.163636e+06,11.0,7500000.0,35200000.0,21600000.0,36500000.0,3,Semi-final
3,2010,France,48.0,26.0,11.0,11.0,67.0,37.0,89.0,0.503080,...,0.770833,275000000.0,2.500000e+07,11.0,16000000.0,83000000.0,68000000.0,108000000.0,0,Group stage
4,2010,Argentina,47.0,28.0,8.0,11.0,81.0,46.0,92.0,0.546102,...,0.978723,273500000.0,2.486364e+07,11.0,6000000.0,44500000.0,75000000.0,148000000.0,2,Quarter-final


worldcup_ml_dataset.csv 저장 완료
